# Experiment 2.2d: Discrete Oracle Per-Layer LR Grid vs Single-LR Muon

**Counterpart script:** `run_experiment.py`

This notebook is intentionally a **serious notebook front-end** for the script, not a second independent implementation. It imports the script, runs its `run_experiment()` API, and then analyzes the returned structured results.

## Truthful scope of this notebook

This is a **toy optimization study** in one fixed regime:

- 4-layer deep linear network, width 32
- synthetic linear regression task
- 300 optimization steps
- rescaling constant `c = 100`
- comparison metric: **final training loss**

The operational question is narrower than the earlier title wording suggested:

> In this fixed `c=100` setting, can an **exhaustive discrete per-layer LR grid** for SGD match or exceed the best **discrete single-LR** Muon baseline on final training loss?

Here **"oracle"** does **not** mean a continuous optimum or a mechanistic proof. It means:

- each layer chooses from a fixed 5-value LR grid,
- all `5^4 = 625` tuples are searched,
- selection uses the first 3 seeds only,
- the selection score is the mean of the **finite** final losses on those seeds.


## Study design and reporting checklist

This notebook presents, at minimum:

1. reproducibility / runtime / configuration information,
2. LR-selection outcomes for SGD, Muon, and the oracle per-layer grid,
3. all-seed and held-out-seed final-loss summaries with finite-count visibility,
4. the selected per-layer oracle LR pattern and initialization gradient-scale context,
5. the current ratio / heuristic outputs used by the script,
6. a calibrated conclusion stating what this comparison does and does not explain.

Because the experiment is moderately expensive (roughly 1900+ training runs), the notebook is designed to run the experiment **once** and then reuse the returned result dictionary for all analysis cells.


In [ ]:
import html
import importlib.util
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Image, Markdown, display

try:
    import pandas as pd
except ImportError:
    pd = None


def display_rows(rows, title=None):
    if title:
        display(Markdown(f"#### {title}"))
    if not rows:
        display(Markdown("_No rows to display._"))
        return None
    if pd is not None:
        df = pd.DataFrame(rows)
        display(df)
        return df

    columns = list(rows[0].keys())
    parts = ["<table>", "<thead><tr>"]
    parts.extend(f"<th>{html.escape(str(col))}</th>" for col in columns)
    parts.append("</tr></thead><tbody>")
    for row in rows:
        parts.append("<tr>")
        for col in columns:
            parts.append(f"<td>{html.escape(str(row.get(col, '')))}</td>")
        parts.append("</tr>")
    parts.append("</tbody></table>")
    display(HTML("".join(parts)))
    return rows


def fmt_number(value):
    if value is None:
        return "None"
    if isinstance(value, str):
        return value
    if not np.isfinite(value):
        return "inf"
    return f"{float(value):.6e}"


def fmt_lr(value):
    if value is None:
        return "None"
    if isinstance(value, (list, tuple)):
        return "[" + ", ".join(f"{float(v):.1e}" for v in value) + "]"
    return f"{float(value):.1e}"


In [ ]:
EXPERIMENT_RELATIVE_PATH = Path(
    "experiments/Tier_2_Symmetry_Reparametrization_Tests/"
    "Exp_2.2_Layerwise_Rescaling_Robustness/"
    "2.2d_Implicit_Per_Layer_LR/run_experiment.py"
)


def locate_experiment_script():
    cwd = Path.cwd().resolve()
    candidate_roots = [cwd, *cwd.parents]

    for root in candidate_roots:
        candidate = root / EXPERIMENT_RELATIVE_PATH
        if candidate.exists():
            return candidate.resolve()

        repo_named_candidate = root / "Muon_as_RG_Gauge_Fixing" / EXPERIMENT_RELATIVE_PATH
        if repo_named_candidate.exists():
            return repo_named_candidate.resolve()

    matches = list(cwd.rglob("2.2d_Implicit_Per_Layer_LR/run_experiment.py"))
    if len(matches) == 1:
        return matches[0].resolve()

    raise FileNotFoundError(
        "Could not locate 2.2d run_experiment.py from the current notebook working directory."
    )


SCRIPT_PATH = locate_experiment_script()
EXPERIMENT_DIR = SCRIPT_PATH.parent
spec = importlib.util.spec_from_file_location("exp_2_2d_implicit_per_layer_lr", SCRIPT_PATH)
exp_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp_module)

display(Markdown(
    f"**Imported script module:** `{SCRIPT_PATH}`  \n"
    f"**Notebook working directory:** `{Path.cwd().resolve()}`"
))


## Reproducibility, runtime, and configuration

The next cell runs the experiment through the script API, records wall-clock runtime, and surfaces the key configuration and seed split. This notebook does **not** duplicate the training logic.


In [ ]:
start_time = time.time()
results = exp_module.run_experiment(save_plot=True, verbose=False, output_dir=exp_module.SCRIPT_DIR)
runtime_seconds = time.time() - start_time

config = results["config"]
seeds = results["seeds"]
plot_info = results["plot"]

display(Markdown(
    "### Run summary\n"
    f"- Runtime: **{runtime_seconds:.2f} s**\n"
    f"- Script entrypoint used: `{SCRIPT_PATH}`\n"
    f"- Saved summary plot: `{plot_info['path']}` (status: `{plot_info['status']}`)\n"
    f"- Total training runs implied by the current protocol: **{config['total_train_runs']}**\n"
    f"- Selection runs: **{config['selection_train_runs']}**; final evaluation runs: **{config['evaluation_train_runs']}**"
))

config_rows = [
    {"field": "dim", "value": config["dim"]},
    {"field": "n_layers", "value": config["n_layers"]},
    {"field": "num_steps", "value": config["num_steps"]},
    {"field": "batch_size", "value": config["batch_size"]},
    {"field": "momentum_beta", "value": config["momentum_beta"]},
    {"field": "ns_iters", "value": config["ns_iters"]},
    {"field": "c", "value": config["c"]},
    {"field": "selection_seed_count", "value": config["selection_seed_count"]},
    {"field": "selection_metric", "value": config["selection_metric"]},
    {"field": "selection_divergence_policy", "value": config["selection_divergence_policy"]},
    {"field": "final_metric", "value": config["final_metric"]},
]
seed_rows = [
    {"seed_role": "all", "value": seeds["all"]},
    {"seed_role": "selection", "value": seeds["selection"]},
    {"seed_role": "held_out", "value": seeds["held_out"]},
]

display_rows(config_rows, title="Configuration")
display_rows(seed_rows, title="Seed split")


## Learning-rate selection outcomes

The script performs three selection procedures:

- a **single global LR sweep** for SGD,
- a **single global LR sweep** for Muon,
- an **exhaustive discrete per-layer LR grid search** for SGD.

The tables and plots below show what was actually selected under the current protocol and how much finite behavior existed during selection.


In [ ]:
sgd_sweep = results["sweeps"]["sgd_single_lr"]
muon_sweep = results["sweeps"]["muon_single_lr"]
oracle_sweep = results["sweeps"]["sgd_oracle_per_layer"]

sgd_rows = [
    {
        "lr": record["lr"],
        "mean_finite_loss": fmt_number(record["mean_finite_loss"]),
        "num_finite": f"{record['num_finite']}/{record['num_total']}",
    }
    for record in sgd_sweep["records"]
]
muon_rows = [
    {
        "lr": record["lr"],
        "mean_finite_loss": fmt_number(record["mean_finite_loss"]),
        "num_finite": f"{record['num_finite']}/{record['num_total']}",
    }
    for record in muon_sweep["records"]
]

oracle_sorted = sorted(
    oracle_sweep["records"],
    key=lambda record: (np.isinf(record["mean_finite_loss"]), record["mean_finite_loss"]),
)
oracle_top_rows = [
    {
        "rank": idx + 1,
        "per_layer_lrs": fmt_lr(record["per_layer_lrs"]),
        "mean_finite_loss": fmt_number(record["mean_finite_loss"]),
        "num_finite": f"{record['num_finite']}/{record['num_total']}",
    }
    for idx, record in enumerate(oracle_sorted[:10])
]

selection_summary_rows = [
    {
        "method": "SGD (single LR)",
        "selection_status": sgd_sweep["selection_status"],
        "selected": fmt_lr(sgd_sweep["best_lr"]),
        "best_selection_mean": fmt_number(sgd_sweep["best_mean_finite_loss"]),
    },
    {
        "method": "Muon (single LR)",
        "selection_status": muon_sweep["selection_status"],
        "selected": fmt_lr(muon_sweep["best_lr"]),
        "best_selection_mean": fmt_number(muon_sweep["best_mean_finite_loss"]),
    },
    {
        "method": "SGD (oracle per-layer LR grid)",
        "selection_status": oracle_sweep["selection_status"],
        "selected": fmt_lr(oracle_sweep["best_per_layer_lrs"]),
        "best_selection_mean": fmt_number(oracle_sweep["best_mean_finite_loss"]),
    },
]

display_rows(selection_summary_rows, title="Selection outcomes")
display_rows(sgd_rows, title="SGD single-LR sweep")
display_rows(muon_rows, title="Muon single-LR sweep")
display_rows(oracle_top_rows, title="Top oracle per-layer LR tuples (best 10 of 625)")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

for ax, sweep, title in [
    (axes[0], sgd_sweep, "SGD single-LR sweep"),
    (axes[1], muon_sweep, "Muon single-LR sweep"),
]:
    xs = [record["lr"] for record in sweep["records"]]
    raw_ys = [record["mean_finite_loss"] for record in sweep["records"]]
    positive_ys = [y for y in raw_ys if np.isfinite(y) and y > 0]
    proxy_y = max(positive_ys) * 10 if positive_ys else 1.0
    plot_ys = [
        y if np.isfinite(y) and y > 0 else proxy_y
        for y in raw_ys
    ]
    ax.plot(xs, plot_ys, marker="o")
    used_proxy = False
    for x, record, plotted_y in zip(xs, sweep["records"], plot_ys):
        y = record["mean_finite_loss"]
        if np.isfinite(y) and y > 0:
            annotation = f"{record['num_finite']}/{record['num_total']}"
        else:
            annotation = f"{record['num_finite']}/{record['num_total']}\nproxy"
            used_proxy = True
        ax.annotate(
            annotation,
            (x, plotted_y),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("candidate LR")
    ax.set_ylabel("selection mean finite loss")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if not positive_ys:
        ax.text(0.5, 0.5, "No finite positive candidates", transform=ax.transAxes, ha="center", va="center", fontsize=11)
    elif used_proxy:
        ax.text(0.98, 0.02, "Proxy markers denote non-positive or non-finite values.", transform=ax.transAxes, ha="right", va="bottom", fontsize=8)

best_oracle_rows = oracle_sorted[:8]
bar_labels = [fmt_lr(record["per_layer_lrs"]) for record in best_oracle_rows][::-1]
raw_bar_values = [record["mean_finite_loss"] for record in best_oracle_rows][::-1]
positive_oracle_values = [value for value in raw_bar_values if np.isfinite(value) and value > 0]
oracle_proxy = max(positive_oracle_values) * 10 if positive_oracle_values else 1.0
bar_values = []
bar_annotations = []
for value in raw_bar_values:
    if np.isfinite(value) and value > 0:
        bar_values.append(value)
        bar_annotations.append(fmt_number(value))
    else:
        bar_values.append(oracle_proxy)
        label = fmt_number(value) if np.isfinite(value) else "inf"
        bar_annotations.append(f"{label} shown at proxy")

bars = axes[2].barh(range(len(bar_labels)), bar_values, color="#FF8800", edgecolor="black")
axes[2].set_yticks(range(len(bar_labels)))
axes[2].set_yticklabels(bar_labels, fontsize=8)
axes[2].set_xscale("log")
axes[2].set_xlabel("selection mean finite loss")
axes[2].set_title("Top oracle LR tuples")
axes[2].grid(True, alpha=0.3, axis="x")
for bar, annotation in zip(bars, bar_annotations):
    axes[2].annotate(
        annotation,
        (bar.get_width(), bar.get_y() + bar.get_height() / 2),
        textcoords="offset points",
        xytext=(5, 0),
        va="center",
        fontsize=8,
    )
if any(not (np.isfinite(value) and value > 0) for value in raw_bar_values):
    axes[2].text(
        0.98,
        0.02,
        "Non-positive or non-finite values are shown at a positive proxy for log scaling.",
        transform=axes[2].transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
plt.show()


## Final evaluation across all seeds

The selected hyperparameters are then evaluated on all 5 seeds. The table below separates:

- **all-seed summaries**, and
- **held-out-only summaries** on the 2 seeds not used for selection.

This is still not a full generalization study, but it makes the evaluation split explicit and visible.


In [ ]:
method_order = [
    ("sgd_single_lr", "SGD (single LR)"),
    ("muon_single_lr", "Muon (single LR)"),
    ("sgd_oracle_per_layer", "SGD (oracle per-layer LR grid)"),
]


def summary_row(method_key, label, subset_key):
    summary = results["evaluations"][method_key][subset_key]
    selected = results["evaluations"][method_key]["selected_hyperparameter"]
    return {
        "method": label,
        "mean_finite_loss": fmt_number(summary["mean_finite_loss"]),
        "std_finite_loss": fmt_number(summary["std_finite_loss"]),
        "finite": f"{summary['num_finite']}/{summary['num_total']}",
        "selected": fmt_lr(selected) if subset_key == "all_seeds" else "--",
    }


all_seed_rows = [summary_row(method_key, label, "all_seeds") for method_key, label in method_order]
held_out_rows = [summary_row(method_key, label, "held_out_seeds") for method_key, label in method_order]

per_seed_rows = []
for method_key, label in method_order:
    for row in results["evaluations"][method_key]["losses_by_seed"]:
        per_seed_rows.append({
            "method": label,
            "seed": row["seed"],
            "loss": fmt_number(row["loss"]),
            "finite": row["finite"],
        })

display_rows(all_seed_rows, title="All-seed final-loss summary")
display_rows(held_out_rows, title="Held-out-only final-loss summary")
display_rows(per_seed_rows, title="Per-seed final losses")

finite_losses = [
    row["loss"]
    for method_key, _ in method_order
    for row in results["evaluations"][method_key]["losses_by_seed"]
    if np.isfinite(row["loss"]) and row["loss"] > 0
]
proxy_loss = max(finite_losses) * 10 if finite_losses else 1.0

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = {
    "SGD (single LR)": "#4477AA",
    "Muon (single LR)": "#CC3311",
    "SGD (oracle per-layer LR grid)": "#FF8800",
}
for x_pos, (method_key, label) in enumerate(method_order):
    rows = results["evaluations"][method_key]["losses_by_seed"]
    xs = [x_pos] * len(rows)
    ys = [row["loss"] if np.isfinite(row["loss"]) and row["loss"] > 0 else proxy_loss for row in rows]
    markers = ["o" if row["finite"] else "x" for row in rows]
    for x, y, marker, row in zip(xs, ys, markers, rows):
        ax.scatter(x, y, color=colors[label], marker=marker, s=70, alpha=0.85)
        ax.annotate(str(row["seed"]), (x, y), textcoords="offset points", xytext=(5, 4), fontsize=8)

ax.set_xticks(range(len(method_order)))
ax.set_xticklabels([label for _, label in method_order], rotation=0)
ax.set_yscale("log")
ax.set_ylabel("Final training loss")
ax.set_title("Per-seed final losses (x marks indicate non-finite runs)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


## Selected oracle per-layer LR pattern and initialization-scale context

The table below reports the initialization gradient norms for the first selection seed alongside the **selected** oracle per-layer LR tuple. This is descriptive context only; it is not a mechanistic proof.

The notebook also displays the saved summary figure generated directly by the script.


In [ ]:
diagnostics_rows = [
    {
        "layer": layer["layer"],
        "gradient_fro_norm_at_init": fmt_number(layer["gradient_fro_norm"]),
        "selected_oracle_lr": fmt_lr(layer["oracle_lr"]),
    }
    for layer in results["initial_diagnostics"]["layers"]
]

display_rows(diagnostics_rows, title="Initialization diagnostics for the first selection seed")

if plot_info["saved"]:
    display(Markdown(f"#### Script-generated summary figure\nSaved at `{plot_info['path']}`"))
    display(Image(filename=plot_info["path"]))
else:
    display(Markdown(f"_Plot was not saved. Plot status: `{plot_info['status']}`._"))


## Operational verdict and final conclusion

The script exposes two ratio-style summaries:

- **oracle / Muon** mean-loss ratio,
- **single-LR SGD / Muon** mean-loss ratio,

plus the current **percent-explained heuristic** when it is actually well-defined.

The key point is to treat these as **toy operational summaries**, not as a full explanation of Muon. The final rendered conclusion below is intentionally calibrated to that scope.


In [ ]:
metrics = results["hypothesis_metrics"]
interpretation = results["interpretation"]

metric_rows = [
    {"metric": "oracle / Muon mean-loss ratio", "value": fmt_number(metrics["ratio_oracle_to_muon"]) + "x"},
    {"metric": "single-LR SGD / Muon mean-loss ratio", "value": fmt_number(metrics["ratio_sgd_to_muon"]) + "x"},
    {"metric": "within-2x verdict", "value": "PASS" if metrics["oracle_matches_muon_within_2x"] else "FAIL"},
    {
        "metric": "percent-explained heuristic",
        "value": (
            f"{metrics['percent_explained_heuristic']:.2f}%"
            if metrics["percent_explained_heuristic"] is not None
            else f"not reported ({metrics['percent_explained_status']})"
        ),
    },
]

display_rows(metric_rows, title="Current ratio / heuristic outputs")

support_lines = "\n".join(f"- {line}" for line in interpretation["summary_lines"])
limitation_lines = "\n".join(f"- {line}" for line in interpretation["caveats"])

display(Markdown(
    "### Verdict\n"
    f"**{interpretation['headline']}**\n\n"
    "**What this notebook can reasonably claim from the current implementation:**\n"
    f"{support_lines}\n\n"
    "**What this notebook cannot claim from the current implementation:**\n"
    f"{limitation_lines}\n\n"
    "### Final conclusion\n"
    f"{interpretation['final_conclusion']}"
))
